In [1]:
from pathlib import Path
from astropy.io import fits
import numpy as np

## A1 (solved)

In [2]:
h = fits.getheader(Path("~/Research/AsTrovello/Output/datacubes/ngc1087/ngc1087_datacube_sci_1135x1277_Jy_per_pixel.fits").expanduser())
print(h.get('CDELT1'), h.get('CDELT2'), h.get('CD1_1'), h.get('PIXAREA'))

3.0807051217028e-05 3.0807051217028e-05 None 0.0123


## A2 (solved)

In [3]:
novo = fits.getdata(Path("~/Research/AsTrovello/Output/datacubes/ngc1087/ngc1087_datacube_sci_1135x1277_Jy_per_pixel.fits").expanduser())
velho = fits.getdata(Path("~/Research/AsTrovello/Output/datacubes/ngc1087/ngc1087_datacube_sci_1134x1276_Jy_per_pixel.fits").expanduser())

nb, ny, nx = velho.shape
print("recortando para:", ny, nx)

n = novo[4, :ny, :nx]
v = velho[4, :ny, :nx]
print("HST f814w, razao novo/velho:", np.nanmedian(n/v))   # esperado ~0.9939

n = novo[12, :ny, :nx]
v = velho[12, :ny, :nx]
print("JWST f2100w, razao:", np.nanmedian(n/v))            # esperado 1.0

FileNotFoundError: [Errno 2] No such file or directory: '/Users/andretrovello/Research/AsTrovello/Output/datacubes/ngc1087/ngc1087_datacube_sci_1134x1276_Jy_per_pixel.fits'

In [4]:
novo = fits.getheader(Path("~/Research/AsTrovello/Output/datacubes/ngc1087/ngc1087_datacube_sci_1135x1277_Jy_per_pixel.fits").expanduser())
print(novo['CDELT1']*3600, np.sqrt(novo['PIXAREA']))   # devem coincidir

0.11090538438130079 0.11090536506409417


In [6]:
h = fits.getheader(Path("~/Research/AsTrovello/Output/datacubes/ngc1087/ngc1087_datacube_sci_1135x1277_Jy_per_pixel.fits").expanduser())
print("CDELT1 :", h.get('CDELT1'))
print("CDELT2 :", h.get('CDELT2'))
print("CD1_1  :", h.get('CD1_1'))
print("PC1_1  :", h.get('PC1_1'))
print("PIXAREA:", h.get('PIXAREA'))
print("escala WCS   :", h.get('CDELT2', 0)*3600, "arcsec/px")
print("escala PIXAREA:", np.sqrt(h.get('PIXAREA', 0)), "arcsec/px")
print("bandas :", sum(1 for k in h if k.startswith('FILT')))
print("shape  :", h['NAXIS3'], h['NAXIS2'], h['NAXIS1'])

CDELT1 : 3.0807051217028e-05
CDELT2 : 3.0807051217028e-05
CD1_1  : None
PC1_1  : -0.99999999991961
PIXAREA: 0.0123
escala WCS   : 0.11090538438130079 arcsec/px
escala PIXAREA: 0.11090536506409417 arcsec/px
bandas : 13
shape  : 13 1277 1135


In [4]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
from pathlib import Path
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
from photutils.aperture import SkyCircularAperture, aperture_photometry
import astropy.units as u

base = Path("~/Research/AsTrovello/Output/datacubes/ngc1087").expanduser()
p_novo  = base / "ngc1087_datacube_sci_1109x1202_Jy_per_pixel.fits"
p_velho = base / "ngc1087_datacube_sci_1135x1277_Jy_per_pixel.fits"

with fits.open(p_novo) as h:
    dn, wn = h[0].data, WCS(h[0].header, naxis=2)
with fits.open(p_velho) as h:
    dv, wv = h[0].data, WCS(h[0].header, naxis=2)

print(f"novo  {dn.shape}   velho {dv.shape}\n")

# aberturas nas fontes mais brilhantes do cubo novo, definidas em coordenadas
# de ceu para que caiam no mesmo lugar nos dois
from scipy.ndimage import maximum_filter, uniform_filter
ref = uniform_filter(np.nan_to_num(dn[4]), size=5)        # f814w
pk = (ref == maximum_filter(ref, size=40)) & np.isfinite(dn[4])
ys, xs = np.where(pk)
ordem = np.argsort(ref[ys, xs])[::-1]
margem = 40
ys, xs = ys[ordem], xs[ordem]
keep = ((ys > margem) & (ys < dn.shape[1]-margem) &
        (xs > margem) & (xs < dn.shape[2]-margem))
ys, xs = ys[keep][:8], xs[keep][:8]
coords = wn.pixel_to_world(xs, ys)
ap = SkyCircularAperture(coords, r=4*u.arcsec)

bandas = ['f275w','f336w','f438w','f555w','f814w','f200w','f300m',
          'f335m','f360m','f770w','f1000w','f1130w','f2100w']
print(f"{'banda':8s}{'razao':>9}{'dispersao':>12}")
print("-"*30)
for i, b in enumerate(bandas):
    fn = np.asarray(aperture_photometry(np.nan_to_num(dn[i]),
                                        ap.to_pixel(wn))['aperture_sum'])
    fv = np.asarray(aperture_photometry(np.nan_to_num(dv[i]),
                                        ap.to_pixel(wv))['aperture_sum'])
    ok = (fv != 0) & np.isfinite(fn) & np.isfinite(fv)
    if ok.sum() == 0:
        print(f"{b:8s}   sem abertura valida"); continue
    r = fn[ok] / fv[ok]
    print(f"{b:8s}{np.median(r):>9.4f}{100*np.std(r):>11.1f}%")

novo  (13, 1202, 1109)   velho (13, 1277, 1135)

banda       razao   dispersao
------------------------------
f275w      0.9883        8.7%
f336w      0.9839       10.9%
f438w      0.9679       18.8%
f555w      0.9661       29.4%
f814w      0.9631        9.7%
f200w      0.9462        7.3%
f300m      0.9570        8.5%
f335m      0.9632       11.4%
f360m      0.9626        7.1%
f770w      0.9908       19.4%
f1000w     0.9898       26.2%
f1130w     0.9898       17.9%
f2100w     0.9963        8.8%
